In [ ]:
import math
import torch
import torch.nn.functional as F

try:
    import triton
    import triton.language as tl
    print(f"PyTorch: {torch.__version__}, Triton: {triton.__version__}")
except ImportError:
    print("Triton is not installed. Please run: pip install triton")

print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

In [ ]:
from naive_attention import memory_footprint_bytes

batch, heads = 2, 8
for seq in [512, 1024, 4096, 8192, 16384, 32768]:
    mb = memory_footprint_bytes(batch, heads, seq, dtype=torch.float16) / (1024 * 1024)
    print(f"Seq Len {seq:5d} -> Attention Matrix P requires: {mb:10.2f} MB ({mb/1024:.2f} GB)")

In [ ]:
from online_softmax import safe_softmax, online_softmax_1d, tiled_online_attention_simulation

x = torch.randn(128)
p_safe = safe_softmax(x)
p_online = online_softmax_1d(x, chunk_size=16)
print("Online Softmax Max Difference:", torch.max(torch.abs(p_safe - p_online)).item())

In [ ]:
from flash_attention import flash_attention
from flash_mha import FlashMHA

print("FlashAttention and FlashMHA successfully loaded!")

In [ ]:
from test_pipeline import run_numerical_test, run_mha_module_test

if torch.cuda.is_available():
    print("--- Testing Causal Attention ---")
    run_numerical_test(batch_size=2, num_heads=4, seq_len=1024, head_dim=64, causal=True)
    print("\n--- Testing Non-Causal Attention ---")
    run_numerical_test(batch_size=2, num_heads=4, seq_len=1024, head_dim=64, causal=False)
    print("\n--- Testing FlashMHA Module ---")
    run_mha_module_test()
else:
    print("CUDA GPU required to run Triton kernel tests. Upload to Colab/Kaggle or run on a GPU host.")

In [ ]:
from benchmark_pipeline import benchmark_suite

if torch.cuda.is_available():
    benchmark_suite(batch_size=2, num_heads=8, head_dim=64, causal=True, seq_lens=(512, 1024, 2048, 4096))
else:
    print("Benchmarking requires an active NVIDIA GPU.")